In [ ]:
# Clear all variables
%reset -f
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from matplotlib.colors import Normalize
from matplotlib import cm
import re
import glob

In [ ]:
input_folder = '../Downloaded_data/EXP_DATA/Irregular_case/Sim_csv_files'
filename = 'SIM_L14_Pos01_Re100_B0001_T.csv'
file_path = input_folder+'/'+filename
output_folder = '../Downloaded_data/EXP_DATA/Irregular_case/Sim_csv_files/Pos1_transformed/'

In [ ]:
input_file_path = os.path.join(input_folder, filename)
name, ext = os.path.splitext(filename)

# Extract layer (L) and position (Pos) information from the filename
layer_match = re.search(r'L(\d{2})', name)
pos_match = re.search(r'Pos(\d{2})', name)

if layer_match and pos_match:
    layer = int(layer_match.group(1))
    pos = int(pos_match.group(1))
    print(f"Layer: {layer}, Position: {pos}")
else:
    print(f"Filename does not contain valid layer or position information: {filename}")

# Since we're working with a CSV file, load it directly with pandas
try:
    df = pd.read_csv(input_file_path)
    print(f"Successfully loaded CSV with {df.shape[0]} rows and {df.shape[1]} columns")
    print(f"Columns: {df.columns.tolist()}")
except Exception as e:
    print(f"Error loading file: {str(e)}")

# Determine rotation angle based on layer
try:
  #  FOR 40 DEGREE CASE (uncomment to use)
    if 'L17' in name or 'L_17' in name:
        alpha = -2 * np.pi / 9     # -40°
    elif 'L16' in name or 'L_16' in name:
        alpha = -4 * np.pi / 9     # -80°
    elif 'L15' in name or 'L_15' in name:
        alpha = -2 * np.pi / 3     # -120°
    elif 'L14' in name or 'L_14' in name:
        alpha = -8 * np.pi / 9     # -160°
    elif 'L13' in name or 'L_13' in name:
        alpha = -10 * np.pi / 9    # -200°  
    print(f"Layer: {layer}, Alpha: {alpha:.4f} rad ({np.degrees(alpha):.1f}°)")
    
except Exception as e:
    print(f"Error setting alpha: {e}")
    alpha = 0
# Determine position parameter 'a'
if 'Pos_01' in name or 'Pos01' in name:
    a = 22.5
elif 'Pos_03' in name or 'Pos03' in name:
    a = 7.5
elif 'Pos_02' in name or 'Pos02' in name:
    a = 15
elif 'Pos_04' in name or 'Pos04' in name:
    a = 0
elif 'Pos_05' in name or 'Pos05' in name:
    a = -7.5
elif 'Pos_06' in name or 'Pos06' in name:
    a = -15
else:  # Assume Pos_07
    a = -22.5
    
cos_alpha = np.cos(alpha)
sin_alpha = np.sin(alpha)

x = 'Points_0'
y = 'Points_1'
z = 'Points_2'
Vel_u = 'av_u_0'
Vel_v = 'av_u_1'
Vel_w = 'av_u_2'
# Transform coordinates
df['x_transformed'] = df[x] * cos_alpha - df[y] * sin_alpha
df['y_transformed'] = df[x] * sin_alpha + df[y] * cos_alpha
df['z_transformed'] = df[z]
# **ADD THIS CENTERING STEP:**
# Center the x-coordinates around zero
x_center = (df['x_transformed'].min() + df['x_transformed'].max()) / 2
df['x_transformed'] = df['x_transformed'] - x_center

# Transform velocity components  
df['Vel_u_transformed'] = df[Vel_u] * cos_alpha - df[Vel_v] * sin_alpha
df['Vel_v_transformed'] = df[Vel_u] * sin_alpha + df[Vel_v] * cos_alpha
df['Vel_w_transformed'] = df[Vel_w]

# Replace original columns with transformed ones
df[x] = df['x_transformed']
df[y] = df['y_transformed'] 
df[z] = df['z_transformed']
df[Vel_u] = df['Vel_u_transformed']
df[Vel_v] = df['Vel_v_transformed']
df[Vel_w] = df['Vel_w_transformed']

# Transform coordinates (assuming you have x, y, z columns)
df['x_transformed'] = df[x] * cos_alpha - df[y] * sin_alpha
df['y_transformed'] = df[x] * sin_alpha + df[y] * cos_alpha
df['z_transformed'] = df[z]  # Z typically doesn't change

# Transform velocity components
df['Vel_u_transformed'] = df[Vel_u] * cos_alpha - df[Vel_v] * sin_alpha
df['Vel_v_transformed'] = df[Vel_u] * sin_alpha + df[Vel_v] * cos_alpha
df['Vel_w_transformed'] = df[Vel_w]  # W component typically doesn't change

# Add alpha and 'a' as columns
df['alpha'] = alpha
df['a'] = a

print(f"Transformation applied: Alpha = {np.degrees(alpha):.1f}°, a = {a}°")

In [ ]:
# === TRANFORMATION ===
df = pd.read_csv(file_path)
df_transformed = df.copy()
df_transformed['x'] = df[x]*np.cos(alpha) + df[y]*np.sin(alpha)
df_transformed['z'] = df[z]
df_transformed['Vel_u'] = df[Vel_u] * np.cos(alpha) + df[Vel_v] * np.sin(alpha)
df_transformed['Vel_w'] = df[Vel_w]

In [ ]:
# Add alpha and a values to the transformed DataFrame
df_transformed['alpha'] = alpha
df_transformed['a'] = a
output_file_path = os.path.join(output_folder, f"{name}_transformed.csv")
df_transformed.to_csv(output_file_path, index=False)
print(f"Transformed file saved at: {output_file_path}")

In [ ]:
# === BATCH PROCESSING (OPTIONAL) ===
# Uncomment and run this cell to process multiple files at once

# def transform_and_save_file(input_folder, filename, output_folder):
#     """Transform a single file and save it"""
#     
#     # Define angle mappings for irregular case
#     layer_angles = {
#         'L17': -5 * np.pi / 9,     # -100°
#         'L16': -11 * np.pi / 36,   # -55°
#         'L15': -31 * np.pi / 36,   # -155°
#         'L14': -np.pi / 36,        # -5°
#         'L13': -np.pi / 2,         # -90°
#         'L12': -11 * np.pi / 18,   # -110°
#         'L11': -13 * np.pi / 36,   # -65°
#         'L10': -5 * np.pi / 12,    # -75°
#         'L09': -7 * np.pi / 12,    # -105°
#         'L08': -13 * np.pi / 18,   # -130°
#         'L07': -np.pi / 4,         # -45°
#         'L06': -11 * np.pi / 36,   # -55°
#         'L05': -17 * np.pi / 36,   # -85°
#         'L04': -np.pi / 12,        # -15°
#         'L03': -11 * np.pi / 18,   # -110°
#         'L02': -11 * np.pi / 18,   # -110°
#         'L01': -7 * np.pi / 12,    # -105°
#     }
#     
#     # Position parameter mapping
#     pos_a_values = {
#         'Pos01': 22.5, 'Pos_01': 22.5,
#         'Pos02': 15, 'Pos_02': 15,
#         'Pos03': 7.5, 'Pos_03': 7.5,
#         'Pos04': 0, 'Pos_04': 0,
#         'Pos05': -7.5, 'Pos_05': -7.5,
#         'Pos06': -15, 'Pos_06': -15,
#         'Pos07': -22.5, 'Pos_07': -22.5,
#     }
#     
#     try:
#         # Load file
#         file_path = os.path.join(input_folder, filename)
#         df = pd.read_csv(file_path)
#         name, ext = os.path.splitext(filename)
#         
#         # Find layer and position
#         layer_match = re.search(r'L(\d{2})', name)
#         pos_match = re.search(r'Pos(\d{2})', name)
#         
#         if not (layer_match and pos_match):
#             return f"SKIP: {filename} - Invalid format"
#         
#         layer_key = f"L{layer_match.group(1)}"
#         
#         # Get alpha
#         alpha = layer_angles.get(layer_key, 0)
#         
#         # Get a
#         a = 0
#         for key in pos_a_values:
#             if key in name:
#                 a = pos_a_values[key]
#                 break
#         
#         # Define columns
#         x_col, y_col, z_col = 'Points_0', 'Points_1', 'Points_2'
#         Vel_u_col, Vel_v_col, Vel_w_col = 'av_u_0', 'av_u_1', 'av_u_2'
#         
#         # Transform
#         df_transformed = df.copy()
#         cos_alpha = np.cos(alpha)
#         sin_alpha = np.sin(alpha)
#         
#         # Store original values
#         x_orig = df[x_col].values
#         y_orig = df[y_col].values
#         z_orig = df[z_col].values
#         Vel_u_orig = df[Vel_u_col].values
#         Vel_v_orig = df[Vel_v_col].values
#         Vel_w_orig = df[Vel_w_col].values
#         
#         # Apply rotation transformation (NO CENTERING)
#         x_rotated = x_orig * cos_alpha + y_orig * sin_alpha
#         y_rotated = -x_orig * sin_alpha + y_orig * cos_alpha
#         z_rotated = z_orig
#         
#         Vel_u_rotated = Vel_u_orig * cos_alpha + Vel_v_orig * sin_alpha
#         Vel_v_rotated = -Vel_u_orig * sin_alpha + Vel_v_orig * cos_alpha
#         Vel_w_rotated = Vel_w_orig
#         
#         df_transformed['x'] = x_rotated
#         df_transformed['y'] = y_rotated
#         df_transformed['z'] = z_rotated
#         df_transformed['Vel_u'] = Vel_u_rotated
#         df_transformed['Vel_v'] = Vel_v_rotated
#         df_transformed['Vel_w'] = Vel_w_rotated
#         df_transformed['Vel_mag'] = np.sqrt(Vel_u_rotated**2 + Vel_w_rotated**2)
#         df_transformed['alpha'] = alpha
#         df_transformed['a'] = a
#         
#         # Save
#         os.makedirs(output_folder, exist_ok=True)
#         output_path = os.path.join(output_folder, f"{name}_transformed.csv")
#         df_transformed.to_csv(output_path, index=False)
#         
#         return f"SUCCESS: {filename} -> {layer_key} ({np.degrees(alpha):.1f}°)"
#         
#     except Exception as e:
#         return f"ERROR: {filename} - {str(e)}"

# # Process all CSV files in the input folder
# all_files = glob.glob(os.path.join(input_folder, '*.csv'))
# 
# print("=" * 60)
# print(f"BATCH PROCESSING: {len(all_files)} files found")
# print("=" * 60)
# 
# results = []
# for file in all_files:
#     filename = os.path.basename(file)
#     result = transform_and_save_file(input_folder, filename, output_folder)
#     results.append(result)
#     print(result)
# 
# print("\n" + "=" * 60)
# print("BATCH PROCESSING COMPLETE")
# print("=" * 60)
# success_count = sum(1 for r in results if r.startswith("SUCCESS"))
# print(f"Successfully processed: {success_count}/{len(results)} files")